# CreaseIQ — Agent Playground (Milestone 1)

Scratch notebook for learning/validating the LangGraph foundation with **mock cricket data** only (no real rules, no RAG yet).

Sections:
1. Setup & Sarvam connection
2. Mock retrieval tool
3. Exercise A — prebuilt `create_agent`
4. Exercise B — custom `StateGraph` (understand → check context → clarify / retrieve → answer)
5. Multi-turn clarification with a checkpointer
6. Languages: English (default) and Assamese, with an optional response-language override
7. A second `create_agent` with the same behaviour as the graph, plus a side-by-side `compare()`

Create `backend/.env` from `.env.example` and set `SARVAM_API_KEY` before running the LLM cells.

## 1. Setup

In [ ]:
import json, os, re
from pathlib import Path
from typing import Literal, Optional, TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# Look for .env in backend/ (parent of src/) or the current directory
load_dotenv(Path.cwd().parent / ".env")
load_dotenv()

SARVAM_BASE_URL = os.getenv("SARVAM_BASE_URL", "https://api.sarvam.ai/v2")
SARVAM_MODEL = os.getenv("SARVAM_MODEL", "sarvam-105b")
print("API key set:", bool(os.getenv("SARVAM_API_KEY")), "| model:", SARVAM_MODEL)
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING", "false"), "| project:", os.getenv("LANGSMITH_PROJECT", "default"),
      "| key set:", bool(os.getenv("LANGSMITH_API_KEY")))

### 1a. Model connection

Per the spec, verify Sarvam's **basic chat** first, and **tool calling separately** (don't assume one implies the other).
Auth header format must be checked against current Sarvam docs — the default below sends the key as a Bearer token via `api_key`; if Sarvam requires an `api-subscription-key` header, uncomment `default_headers`.

In [ ]:
llm = ChatOpenAI(
    model=SARVAM_MODEL,
    base_url=SARVAM_BASE_URL,
    api_key=os.getenv("SARVAM_API_KEY", "missing"),
    # default_headers={"api-subscription-key": os.getenv("SARVAM_API_KEY", "")},
    temperature=0,
    timeout=60,
    max_retries=1,
)

In [ ]:
# Test 1: basic text completion
print(llm.invoke("In one sentence, what is a no-ball in cricket?").content)

In [ ]:

# Test 2: Assamese - does the model reply in Assamese script, and render cricket terms sensibly?
print(llm.invoke("ক্ৰিকেটত নো-বল মানে কি? অসমীয়াত এটা বাক্যত উত্তৰ দিয়ক।").content)


## 2. Mock retrieval tool
Stand-in for `search_cricket_rules`. Returns canned, **fake** passages — not real MCC/ICC text.

In [ ]:
MOCK_RULES = [
    {"source": "MOCK Laws of Cricket", "law": "21", "title": "No ball",
     "text": "MOCK: A delivery is a no ball if the bowler's front foot lands beyond the popping crease."},
    {"source": "MOCK Laws of Cricket", "law": "30", "title": "Bowled",
     "text": "MOCK: The striker is out bowled if the wicket is put down by a ball delivered by the bowler."},
    {"source": "MOCK ICC T20I Playing Conditions", "law": "PC-4", "title": "Fielding restrictions",
     "text": "MOCK: During the powerplay a maximum of two fielders are permitted outside the inner circle."},
]

@tool
def search_cricket_rules(query: str, format: Optional[str] = None,
                         competition: Optional[str] = None, match_date: Optional[str] = None) -> str:
    """Search the cricket rules knowledge base. `format` is 'odi' or 't20i'; `competition` and `match_date` (ISO) are optional
    context (ignored by this mock)."""
    words = set(re.findall(r"\w+", query.lower()))
    hits = [r for r in MOCK_RULES if words & set(re.findall(r"\w+", (r["title"] + " " + r["text"]).lower()))]
    return json.dumps(hits[:3]) if hits else "NO_RESULTS"

print(search_cricket_rules.invoke({"query": "no ball front foot"}))

## 3. Exercise A — prebuilt `create_agent`
Learning only. Production uses the custom graph in section 4. This also tests Sarvam **tool calling** and multi-turn tool-call messages.

In [ ]:
from langchain.agents import create_agent

simple_agent = create_agent(
    llm,
    tools=[search_cricket_rules],
    system_prompt="You are a cricket rules assistant. Use the tool, answer briefly, cite the law number. Never invent rules.",
)

result = simple_agent.invoke({"messages": [HumanMessage("Is it a no ball if the front foot is over the line?")]})
for m in result["messages"]:
    m.pretty_print()

## 4. Exercise B — custom `StateGraph`

```
START → understand_question → check_context ─┬─ unsupported ──────────────────────────────→ handle_unsupported → END
                                             ├─ missing facts (≤ MAX_CLARIFICATIONS) ─────→ ask_clarification → END
                                             └─ ok → retrieve_rules → analyse_scenario → validate_answer ─┬─ passed → generate_answer → END
                                                                                                          └─ failed → handle_unsupported → END
any node error ─────────────────────────────────────────────────────────────────────────────→ handle_error → END
```
- `analyse_scenario` compares the facts with the retrieved evidence and proposes citations.
- `validate_answer` is deterministic: it keeps only citations that map to a retrieved passage; if none survive it abstains (`insufficient_evidence`).
- `generate_answer` writes the reply; the **Rule reference** line is built in code from the validated citations, so it cannot be invented.
- `handle_error` returns a safe message; the internal error tag stays in state / LangSmith traces.
- **Clarification cap:** at most `MAX_CLARIFICATIONS` (2) questions per user question. If facts are still missing after that, the graph continues and the answer states which facts are unknown. The counter resets when a question is answered, refused, or found to lack evidence.
- The LLM is asked for JSON as plain text (not structured-output / tool calling), so the nodes don't depend on those Sarvam features.

In [ ]:
class CreaseState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    question: str
    requested_language: Optional[str]  # optional caller preference (e.g. from the API/UI); None = follow the user
    input_language: str              # detected: en | as | other
    response_language: str           # language the answer is written in: en | as
    search_query_en: str             # English retrieval query (rules sources are English)
    intent: str                      # rule_lookup | scenario | comparison | follow_up | out_of_scope
    format: str                      # odi | t20i | unknown
    competition: Optional[str]       # competition named by the user, if any
    match_date: Optional[str]        # ISO date given by the user, if any
    scenario_facts: list[str]
    missing_information: list[str]
    clarification_question: Optional[str]
    clarification_count: int         # questions asked for the current question; reset once it is resolved
    retrieved_rules: list[dict]
    analysis_summary: str            # concise, evidence-based analysis (English) from analyse_scenario
    citations: list                  # analyse_scenario proposes evidence numbers; validate_answer replaces them with verified [{source, law, title}]
    validation_status: str           # passed | failed (set by validate_answer)
    unsupported_reason: Optional[str]  # out_of_scope | insufficient_evidence
    error: Optional[str]             # safe internal error tag, e.g. 'analyse_scenario:TimeoutError' (never shown to users)
    answer: str
    status: str                      # answered | clarification_required | unsupported | insufficient_evidence | error

In [ ]:
# ---- Language support -------------------------------------------------------
# English is the default/fallback. Add a new language by adding one entry to each dict.
SUPPORTED_LANGUAGES = {"en": "English", "as": "Assamese (অসমীয়া)"}
DEFAULT_LANGUAGE = "en"

# Max clarification questions per user question. After this the graph answers with the unknown facts
# called out (never guesses them) instead of asking forever.
MAX_CLARIFICATIONS = 2

# Fixed (non-LLM) messages. NOTE: Assamese strings need review by a native speaker.
STRINGS = {
    "en": {
        "unsupported": "I can only help with men's ODI and T20I cricket rules.",
        "no_evidence": "I couldn't verify this from the available sources.",
        "error": "Something went wrong while processing your question. Please try again.",
    },
    "as": {
        "unsupported": "মই কেৱল পুৰুষ ODI আৰু T20I ক্ৰিকেটৰ নিয়মৰ বিষয়ে সহায় কৰিব পাৰোঁ।",
        "no_evidence": "উপলব্ধ উৎসসমূহৰ পৰা মই ইয়াক নিশ্চিত কৰিব নোৱাৰিলোঁ।",
        "error": "আপোনাৰ প্ৰশ্নটো প্ৰক্ৰিয়া কৰোঁতে কিবা ভুল হ'ল। অনুগ্ৰহ কৰি আকৌ চেষ্টা কৰক।",
    },
}

def lang_instruction(code: str) -> str:
    name = SUPPORTED_LANGUAGES.get(code, SUPPORTED_LANGUAGES[DEFAULT_LANGUAGE])
    return (f"Write your reply in {name}. Keep law numbers, clause IDs, source titles and quoted rule text exactly as given "
            f"(do not translate or transliterate them); translate only your own explanation. "
            f"Do not claim the reply is an official translation.")

def pick_response_language(requested: Optional[str], detected: str) -> str:
    """Caller preference wins; otherwise mirror the user's language; unsupported -> English."""
    for code in (requested, detected):
        if code in SUPPORTED_LANGUAGES:
            return code
    return DEFAULT_LANGUAGE

# ---- Nodes ------------------------------------------------------------------
import functools, logging
log = logging.getLogger("creaseiq")

def safe_node(fn):
    """Turn an unexpected exception inside a node into state['error'] so the graph routes to handle_error."""
    @functools.wraps(fn)
    def wrapper(state):
        try:
            return fn(state)
        except Exception as e:  # LLM timeout, bad response, retrieval failure, ...
            log.warning("node %s failed: %s", fn.__name__, type(e).__name__)  # type only: messages may contain secrets
            return {"error": f"{fn.__name__}:{type(e).__name__}"}
    return wrapper

def clean_optional(v) -> Optional[str]:
    return None if v is None or str(v).strip().lower() in ("", "null", "none", "unknown", "n/a") else str(v).strip()

def format_evidence(rules: list[dict]) -> str:
    return "\n".join(f"[{i}] ({r['source']} | Law {r['law']}) {r['text']}" for i, r in enumerate(rules, 1))

UNDERSTAND_PROMPT = """You analyse cricket-rules questions. The user may write in English or Assamese (Assamese script or romanised). Given the conversation, reply with ONLY a JSON object:
{
  "input_language": "en" | "as" | "other",
  "intent": "rule_lookup" | "scenario" | "comparison" | "follow_up" | "out_of_scope",
  "format": "odi" | "t20i" | "unknown",
  "competition": "competition/series name the user mentioned, or null",
  "match_date": "ISO date (YYYY-MM-DD) the user mentioned, or null",
  "scenario_facts": [facts the user explicitly stated, accumulated across the conversation, written in English],
  "missing_information": [facts that would materially change the answer and are NOT yet known, written in English],
  "search_query_en": "a short English query describing the rule being asked about"
}
Rules: input_language is the language of the LATEST user message; do not assume a format, competition or date that was not stated; do not turn assumptions into facts; only list a missing fact if it could change the outcome.
Only men's ODI and T20I cricket are supported; anything else is out_of_scope."""

def parse_json(text: str) -> dict:
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group(0)) if m else {}

@safe_node
def understand_question(state: CreaseState) -> CreaseState:
    resp = llm.invoke([SystemMessage(UNDERSTAND_PROMPT), *state["messages"]])
    try:
        data = parse_json(resp.content)
    except json.JSONDecodeError:
        data = {}
    question = state["messages"][-1].content
    detected = data.get("input_language", DEFAULT_LANGUAGE)
    return {
        "question": question,
        "input_language": detected,
        "response_language": pick_response_language(state.get("requested_language"), detected),
        "intent": data.get("intent", "rule_lookup"),
        "format": data.get("format", "unknown"),
        "competition": clean_optional(data.get("competition")),
        "match_date": clean_optional(data.get("match_date")),
        "scenario_facts": data.get("scenario_facts", []),
        "missing_information": data.get("missing_information", []),
        "search_query_en": data.get("search_query_en") or question,
        # reset per-turn outputs so nothing stale leaks in from the previous turn of this thread
        "retrieved_rules": [], "analysis_summary": "", "citations": [], "validation_status": "",
        "unsupported_reason": None, "clarification_question": None, "error": None,
    }

def check_context(state: CreaseState) -> CreaseState:
    # Routing happens in route_after_check; this node is a hook for extra deterministic checks.
    return {}

@safe_node
def ask_clarification(state: CreaseState) -> CreaseState:
    missing = state["missing_information"][0]  # one focused question at a time
    q = llm.invoke([
        SystemMessage("Ask ONE short, focused question to get this missing cricket fact. Do not repeat the scenario. "
                      + lang_instruction(state["response_language"])),
        HumanMessage(f"Scenario facts: {state.get('scenario_facts')}\nMissing: {missing}"),
    ]).content
    return {"clarification_question": q, "answer": q, "status": "clarification_required",
            "clarification_count": state.get("clarification_count", 0) + 1,
            "messages": [AIMessage(q)]}

@safe_node
def retrieve_rules(state: CreaseState) -> CreaseState:
    # Sources are English, so always search with the English query.
    raw = search_cricket_rules.invoke({
        "query": state["search_query_en"], "format": state.get("format"),
        "competition": state.get("competition"), "match_date": state.get("match_date"),
    })
    return {"retrieved_rules": [] if raw == "NO_RESULTS" else json.loads(raw)}

ANALYSE_PROMPT = """You compare a cricket scenario with rule evidence. Reply with ONLY a JSON object:
{
  "summary": "1-3 sentences in English: how the evidence applies to the stated facts. Do not assume unstated facts.",
  "citations": [<the NUMBERS of the evidence items you relied on, e.g. 1 or 1, 2>]
}
Use ONLY the evidence. Cite by number only (never copy evidence text into citations). If the evidence does not address the question, return an empty citations list."""

@safe_node
def analyse_scenario(state: CreaseState) -> CreaseState:
    rules = state.get("retrieved_rules", [])
    if not rules:
        return {"analysis_summary": "", "citations": []}
    resp = llm.invoke([
        SystemMessage(ANALYSE_PROMPT),
        HumanMessage(f"Facts: {state.get('scenario_facts')}\nQuestion: {state['question']}\nEvidence:\n{format_evidence(rules)}"),
    ])
    try:
        data = parse_json(resp.content)
    except json.JSONDecodeError:
        data = {}  # unparsable analysis -> no citations -> validation fails -> safe abstention
    return {"analysis_summary": str(data.get("summary", "")), "citations": data.get("citations", [])}

def validate_answer(state: CreaseState) -> CreaseState:
    """Deterministic check: keep only citations that map to a retrieved passage; abstain if none survive."""
    rules = state.get("retrieved_rules", [])
    valid, seen = [], set()
    for c in state.get("citations", []):          # analyse_scenario proposes evidence numbers (1-based)
        try:
            n = int(c)
        except (TypeError, ValueError):
            continue                               # not a number -> cannot map to evidence -> dropped
        if 1 <= n <= len(rules) and n not in seen:  # a number outside the retrieved set is a fabricated citation
            seen.add(n)
            r = rules[n - 1]
            valid.append({"source": r["source"], "law": str(r["law"]), "title": r.get("title")})
    if not valid or not state.get("analysis_summary"):
        return {"validation_status": "failed", "citations": [], "unsupported_reason": "insufficient_evidence"}
    return {"validation_status": "passed", "citations": valid}

@safe_node
def generate_answer(state: CreaseState) -> CreaseState:
    lang = state["response_language"]
    unknown = state.get("missing_information", [])  # non-empty here only if the clarification cap was hit
    unknown_note = (f" These facts are STILL UNKNOWN: {unknown}. Do not assume them. Answer for the stated facts only, say which "
                    f"unknown fact could change the outcome, and advise checking with the on-field umpire.") if unknown else ""
    scope_note = (f" The user mentioned a competition/date ({state.get('competition')}, {state.get('match_date')}). Competition-specific "
                  f"playing conditions could NOT be verified from the available sources, so say this is the general MCC/ICC rule, "
                  f"not competition-specific.") if (state.get("competition") or state.get("match_date")) else ""
    text = llm.invoke([
        SystemMessage("Write the final reply concisely from the analysis: state the outcome, then a one-line reason. Do NOT write a rule "
                      "reference (it is added separately) and do not mention any law that is not in the analysis. Add a short note that "
                      "this is decision support, not an official umpire decision. " + unknown_note + scope_note + lang_instruction(lang)),
        HumanMessage(f"Question: {state['question']}\nFacts: {state.get('scenario_facts')}\nAnalysis: {state['analysis_summary']}"),
    ]).content
    refs = "; ".join(f"[{c['source']} | Law {c['law']}]" for c in state["citations"])  # built in code: cannot be fabricated
    ans = f"{text.strip()}\n\nRule reference: {refs}"
    return {"answer": ans, "status": "answered", "clarification_count": 0, "messages": [AIMessage(ans)]}

def handle_unsupported(state: CreaseState) -> CreaseState:
    reason = state.get("unsupported_reason") or "out_of_scope"
    lang = state.get("response_language", DEFAULT_LANGUAGE)
    if reason == "insufficient_evidence":
        msg, status = STRINGS[lang]["no_evidence"], "insufficient_evidence"
    else:
        msg, status = STRINGS[lang]["unsupported"], "unsupported"
    return {"answer": msg, "status": status, "unsupported_reason": reason, "clarification_count": 0,
            "messages": [AIMessage(msg)]}

def handle_error(state: CreaseState) -> CreaseState:
    """Safe, user-friendly failure: the internal error tag stays in state/traces, never in the reply."""
    msg = STRINGS[state.get("response_language", DEFAULT_LANGUAGE)]["error"]
    return {"answer": msg, "status": "error", "messages": [AIMessage(msg)]}


In [ ]:
def route_after_check(state: CreaseState) -> Literal["handle_error", "handle_unsupported", "ask_clarification", "retrieve_rules"]:
    if state.get("error"):
        return "handle_error"
    if state["intent"] == "out_of_scope":
        return "handle_unsupported"
    if state["missing_information"] and state.get("clarification_count", 0) < MAX_CLARIFICATIONS:
        return "ask_clarification"
    return "retrieve_rules"

def route_after_validate(state: CreaseState) -> Literal["handle_error", "generate_answer", "handle_unsupported"]:
    if state.get("error"):
        return "handle_error"
    return "generate_answer" if state.get("validation_status") == "passed" else "handle_unsupported"

def continue_or_error(next_node: str):
    """Edge router: go to next_node unless the previous node recorded an error."""
    def route(state: CreaseState) -> str:
        return "handle_error" if state.get("error") else next_node
    return route

builder = StateGraph(CreaseState)
for name, fn in [("understand_question", understand_question), ("check_context", check_context),
                 ("ask_clarification", ask_clarification), ("retrieve_rules", retrieve_rules),
                 ("analyse_scenario", analyse_scenario), ("validate_answer", validate_answer),
                 ("generate_answer", generate_answer), ("handle_unsupported", handle_unsupported),
                 ("handle_error", handle_error)]:
    builder.add_node(name, fn)

builder.add_edge(START, "understand_question")
builder.add_conditional_edges("understand_question", continue_or_error("check_context"), ["check_context", "handle_error"])
builder.add_conditional_edges("check_context", route_after_check)
builder.add_conditional_edges("ask_clarification", continue_or_error(END), [END, "handle_error"])
builder.add_conditional_edges("retrieve_rules", continue_or_error("analyse_scenario"), ["analyse_scenario", "handle_error"])
builder.add_conditional_edges("analyse_scenario", continue_or_error("validate_answer"), ["validate_answer", "handle_error"])
builder.add_conditional_edges("validate_answer", route_after_validate)
builder.add_conditional_edges("generate_answer", continue_or_error(END), [END, "handle_error"])
for n in ("handle_unsupported", "handle_error"):
    builder.add_edge(n, END)

graph = builder.compile(checkpointer=InMemorySaver())  # in-memory: local experiments only
graph

## 5. Run it (single turn + multi-turn clarification)
`thread_id` scopes the conversation state.

In [ ]:
def ask(text: str, thread_id: str = "demo-1", language: Optional[str] = None) -> CreaseState:
    """`language` = optional response-language override ('en' / 'as'); None = reply in the user's language."""
    out = graph.invoke(
        {"messages": [HumanMessage(text)], "requested_language": language},  # reset each turn
        config={"configurable": {"thread_id": thread_id}},
    )
    print(f"[{out['status']} | in={out.get('input_language', '?')} out={out.get('response_language', '?')}] {out['answer']}\n")
    return out

_ = ask("Is it a no ball if the bowler's front foot lands beyond the popping crease in a T20I?", "t1")

In [ ]:
# Clarification loop: first turn should ask, second turn (same thread) should answer
_ = ask("The batter left the crease and the ball hit the stumps. Is he out?", "t2")
_ = ask("It was an ODI and the ball was live.", "t2")

In [ ]:
# Out of scope
_ = ask("Who won the 2011 football world cup?", "t3")

### Assamese examples
The reply should follow the user's language. Pass `language=` to override (e.g. ask in Assamese, answer in English).

In [ ]:

# Assamese question -> Assamese answer (law numbers / source titles stay unchanged)
_ = ask("T20I ত বলাৰৰ সন্মুখৰ ভৰি পপিং ক্ৰীজৰ বাহিৰত পৰিলে সেয়া নো-বল নেকি?", "a1")


In [ ]:

# Assamese clarification loop (same thread)
_ = ask("বেটাৰে ক্ৰীজ এৰিলে আৰু বলটোৱে ষ্টাম্পত লাগিল। তেওঁ আউট নেকি?", "a2")
_ = ask("সেয়া ODI আছিল আৰু বলটো লাইভ আছিল।", "a2")


In [ ]:

# Assamese question, English answer (caller override)
_ = ask("T20I ত বলাৰৰ সন্মুখৰ ভৰি পপিং ক্ৰীজৰ বাহিৰত পৰিলে সেয়া নো-বল নেকি?", "a3", language="en")

# English question, Assamese answer
_ = ask("Is it a no ball if the front foot lands beyond the popping crease in a T20I?", "a4", language="as")

# Assamese out-of-scope -> fixed Assamese message
_ = ask("২০১১ চনৰ ফুটবল বিশ্বকাপ কোনে জিকিছিল?", "a5")


In [ ]:
# Inspect the saved state for a thread
graph.get_state({"configurable": {"thread_id": "t2"}}).values


## 6. `create_agent` with the same behaviour as the graph

Same goals as the `StateGraph` (section 4), but there are no nodes or routing code: the **system prompt carries the rules** and the **model decides** when to ask, search or refuse.
It reuses the same tool (`search_cricket_rules`), the same languages and the same fixed refusal / no-evidence strings, so the two can be compared fairly.

Differences to watch for in the comparison:
- The clarification cap is only a **prompt instruction** here (the graph enforces it in code).
- There is no `status` field; the status is inferred from what the agent did.
- Tool calls are chosen by the model, so you can see whether Sarvam's tool calling is reliable.


In [ ]:
from langchain.agents import create_agent
def build_agent_prompt() -> str:
    refusals = "\n".join(f"  - {SUPPORTED_LANGUAGES[c]}: {s['unsupported']}" for c, s in STRINGS.items())
    no_ev = "\n".join(f"  - {SUPPORTED_LANGUAGES[c]}: {s['no_evidence']}" for c, s in STRINGS.items())
    langs = ", ".join(SUPPORTED_LANGUAGES.values())
    return f"""You are CreaseIQ, a cricket rules assistant (decision support, not an official umpire).

SCOPE: men's ODI and T20I cricket rules only.
- If the question is outside this scope, reply with exactly this fixed message in the user's language and do not call any tool:
{refusals}

LANGUAGE: Supported languages: {langs}. Reply in the language of the user's latest message (English if unsupported).
If a message starts with "[Reply language: X]", reply in X instead. Keep law numbers, clause IDs, source titles and quoted rule text
exactly as given (never translate or transliterate them); translate only your own explanation. Never claim a reply is an official translation.

CLARIFY: If a fact that could change the outcome is missing (including the match format when it matters), ask ONE short, focused question
and call no tool. Do not repeat the scenario and do not ask for facts already given. Ask at most {MAX_CLARIFICATIONS} questions for the same
user question; after that, answer for the stated facts only, say which unknown fact could change the outcome, and advise checking with the
on-field umpire. Never guess or assume a missing fact.

ANSWER: Otherwise call `search_cricket_rules` with a short ENGLISH query (the rule sources are English) and the format ('odi' or 't20i') if known.
- If it returns NO_RESULTS, reply with exactly this fixed message in the user's language:
{no_ev}
- Otherwise answer concisely using ONLY the tool results: the outcome, a one-line reason, then "Rule reference: [source | Law X]".
  Cite only laws that appear in the tool results. Add a short note that this is decision support, not an official umpire decision."""

same_agent = create_agent(
    llm,
    tools=[search_cricket_rules],
    system_prompt=build_agent_prompt(),
    checkpointer=InMemorySaver(),   # multi-turn memory per thread_id, like the graph
)

def ask_agent(text: str, thread_id: str = "agent-1", language: Optional[str] = None) -> dict:
    """Same call shape as `ask()`; prints which tools the model called this turn."""
    content = f"[Reply language: {SUPPORTED_LANGUAGES[language]}] {text}" if language else text
    out = same_agent.invoke({"messages": [HumanMessage(content)]}, config={"configurable": {"thread_id": thread_id}})
    msgs = out["messages"]
    last_human = max(i for i, m in enumerate(msgs) if m.type == "human")
    calls = [f"{tc['name']}({tc['args']})" for m in msgs[last_human:] if m.type == "ai" for tc in (m.tool_calls or [])]
    answer = msgs[-1].content
    kind = "tool_used" if calls else "no_tool (clarify / refuse / direct)"
    print(f"[{kind}]" + (f" tools={calls}" if calls else ""), "\n" + answer + "\n")
    return {"answer": answer, "tool_calls": calls}


In [ ]:

def compare(turns: list[str], thread: str, language: Optional[str] = None):
    """Run the same conversation through the StateGraph and the create_agent version."""
    for name, fn in (("STATEGRAPH", ask), ("CREATE_AGENT", ask_agent)):
        print("=" * 20, name, "=" * 20)
        for t in turns:
            print(">", t)
            fn(t, f"{thread}-{name.lower()}", language)

compare(["Is it a no ball if the bowler's front foot lands beyond the popping crease in a T20I?"], "cmp1")


In [ ]:

# Clarification + Assamese + out-of-scope, same inputs for both
compare(["The batter left the crease and the ball hit the stumps. Is he out?", "It was an ODI and the ball was live."], "cmp2")
compare(["T20I ত বলাৰৰ সন্মুখৰ ভৰি পপিং ক্ৰীজৰ বাহিৰত পৰিলে সেয়া নো-বল নেকি?"], "cmp3")
compare(["২০১১ চনৰ ফুটবল বিশ্বকাপ কোনে জিকিছিল?"], "cmp4")
